# Import Data to MySQL - Simple Version

## ⚙️ ก่อนเริ่ม:
1. ✅ MySQL รันอยู่
2. ✅ Database `portfolio_backtesting` มีอยู่แล้ว
3. ✅ Tables 9 ตัวสร้างแล้ว (รัน complete_setup.sql)
4. ✅ รู้ path ของโฟลเดอร์ desktop-tutorial

## 📋 จะ import:
- ETF Master (50 rows)
- Benchmark Portfolios (35 rows)
- Benchmark Holdings (117 rows)
- Price History (208,700 rows)

## 🎯 Step 1: ตั้งค่า Path ของคุณ

**⚠️ สำคัญ: แก้ไขบรรทัดด้านล่างให้เป็น path ของคุณ**

In [ ]:
# ========================================
# แก้ไขตรงนี้! ใส่ path ของโฟลเดอร์ desktop-tutorial
# ========================================

# สำหรับ Windows (ใช้ r'' หรือ \\ ):
PROJECT_PATH = r'C:\Users\YourName\desktop-tutorial'

# สำหรับ Mac/Linux:
# PROJECT_PATH = '/Users/YourName/desktop-tutorial'

# MySQL Password (แก้ถ้าต้องการ):
MYSQL_PASSWORD = 'krittanut123456'

# ========================================
# ห้ามแก้ด้านล่างนี้
# ========================================
import os
os.chdir(PROJECT_PATH)
print(f"✅ Working directory: {os.getcwd()}")
print(f"\n📁 Checking files...")

files = [
    'data/etf_list.csv',
    'data/benchmark_portfolios.csv',
    'data/benchmark_holdings.csv',
    'data/etf_price_history.csv'
]

for f in files:
    exists = os.path.exists(f)
    print(f"   {'✅' if exists else '❌'} {f}")
    if not exists and 'price_history' in f:
        print("\n⚠️  Missing price_history.csv")
        print("   Run: python scripts/generate_sample_data.py")

## 📦 Step 2: Import Libraries

In [ ]:
import mysql.connector
from mysql.connector import Error
import pandas as pd

print("✅ Libraries imported")

## 🔌 Step 3: Connect to MySQL

In [ ]:
# Connect to MySQL
try:
    connection = mysql.connector.connect(
        host='127.0.0.1',
        port=3306,
        user='root',
        password=MYSQL_PASSWORD,
        database='portfolio_backtesting'
    )
    cursor = connection.cursor(dictionary=True)
    print("✅ Connected to MySQL")
    
    # Check tables
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()
    print(f"\n📊 Found {len(tables)} tables:")
    for t in tables:
        print(f"   - {list(t.values())[0]}")
        
except Error as e:
    print(f"❌ Error: {e}")
    print("\nกรุณาตรวจสอบ:")
    print("  1. MySQL รันอยู่หรือไม่?")
    print("  2. Password ถูกต้องหรือไม่?")
    print("  3. Database 'portfolio_backtesting' มีหรือไม่?")

## 📥 Step 4: Import ETF Master (50 rows)

In [ ]:
print("📥 Importing ETF Master...\n")

# Read CSV
df = pd.read_csv('data/etf_list.csv')
print(f"Found {len(df)} ETFs")
display(df.head())

# Clear old data
cursor.execute("DELETE FROM etf_master")
connection.commit()

# Import
for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO etf_master (ticker_symbol, etf_name, asset_class, region, sector, expense_ratio, inception_date)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (
        row['ticker_symbol'],
        row['etf_name'],
        row['asset_class'],
        row.get('region'),
        row.get('sector'),
        float(row['expense_ratio']) if pd.notna(row.get('expense_ratio')) else None,
        row.get('inception_date') if pd.notna(row.get('inception_date')) else None
    ))

connection.commit()

# Verify
cursor.execute("SELECT COUNT(*) as count FROM etf_master")
count = cursor.fetchone()['count']
print(f"\n✅ Imported {count} ETFs")

## 📥 Step 5: Import Benchmark Portfolios (35 rows)

In [ ]:
print("📥 Importing Benchmark Portfolios...\n")

# Read CSV
df = pd.read_csv('data/benchmark_portfolios.csv')
print(f"Found {len(df)} benchmarks")
display(df.head())

# Clear old data
cursor.execute("DELETE FROM benchmark_portfolios")
connection.commit()

# Import
for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO benchmark_portfolios (benchmark_name, description, risk_level, target_return, asset_allocation)
        VALUES (%s, %s, %s, %s, %s)
    """, (
        row['benchmark_name'],
        row.get('description'),
        row.get('risk_level', 'Moderate'),
        float(row['target_return']) if pd.notna(row.get('target_return')) else None,
        row.get('asset_allocation')
    ))

connection.commit()

# Verify
cursor.execute("SELECT COUNT(*) as count FROM benchmark_portfolios")
count = cursor.fetchone()['count']
print(f"\n✅ Imported {count} benchmarks")

## 📥 Step 6: Import Benchmark Holdings (117 rows)

In [ ]:
print("📥 Importing Benchmark Holdings...\n")

# Get mappings
cursor.execute("SELECT benchmark_id, benchmark_name FROM benchmark_portfolios")
benchmark_map = {r['benchmark_name']: r['benchmark_id'] for r in cursor.fetchall()}

cursor.execute("SELECT etf_id, ticker_symbol FROM etf_master")
etf_map = {r['ticker_symbol']: r['etf_id'] for r in cursor.fetchall()}

# Read CSV
df = pd.read_csv('data/benchmark_holdings.csv')
print(f"Found {len(df)} holdings")
display(df.head())

# Clear old data
cursor.execute("DELETE FROM benchmark_holdings")
connection.commit()

# Import
count = 0
for _, row in df.iterrows():
    benchmark_id = benchmark_map.get(row['benchmark_name'])
    etf_id = etf_map.get(row['ticker_symbol'])
    
    if benchmark_id and etf_id:
        cursor.execute("""
            INSERT INTO benchmark_holdings (benchmark_id, etf_id, target_weight)
            VALUES (%s, %s, %s)
        """, (benchmark_id, etf_id, float(row['target_weight'])))
        count += 1

connection.commit()

# Verify
cursor.execute("SELECT COUNT(*) as count FROM benchmark_holdings")
total = cursor.fetchone()['count']
print(f"\n✅ Imported {total} holdings")

## 📥 Step 7: Import Price History (208,700 rows) ⏱️ 1-2 นาที

In [ ]:
print("📥 Importing Price History (ใช้เวลา 1-2 นาที)...\n")

# Get ETF mapping
cursor.execute("SELECT etf_id, ticker_symbol FROM etf_master")
etf_map = {r['ticker_symbol']: r['etf_id'] for r in cursor.fetchall()}

# Clear old data
cursor.execute("DELETE FROM price_history")
connection.commit()
print("Cleared old data\n")

# Import in chunks
total = 0
for chunk in pd.read_csv('data/etf_price_history.csv', chunksize=10000):
    chunk['etf_id'] = chunk['ticker_symbol'].map(etf_map)
    chunk = chunk.dropna(subset=['etf_id'])
    
    batch = []
    for _, row in chunk.iterrows():
        batch.append((
            int(row['etf_id']),
            row['date'],
            float(row['open']),
            float(row['high']),
            float(row['low']),
            float(row['close']),
            float(row['adj_close']),
            int(float(row['volume']))
        ))
    
    if batch:
        cursor.executemany("""
            INSERT INTO price_history (etf_id, date, open, high, low, close, adj_close, volume)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, batch)
        connection.commit()
        total += len(batch)
        print(f"Imported {total:,} records...", end='\r')

# Verify
cursor.execute("SELECT COUNT(*) as count FROM price_history")
count = cursor.fetchone()['count']
print(f"\n\n✅ Imported {count:,} price records")

## ✅ Step 8: Summary - ตรวจสอบทุกอย่าง

In [ ]:
print("="*70)
print("IMPORT SUMMARY".center(70))
print("="*70)
print()

cursor.execute("SHOW TABLES")
tables = [list(t.values())[0] for t in cursor.fetchall()]

summary = []
for table in tables:
    cursor.execute(f"SELECT COUNT(*) as count FROM {table}")
    count = cursor.fetchone()['count']
    summary.append({'Table': table, 'Rows': count})
    print(f"   {table:30s} {count:>10,} rows")

print("\n" + "="*70)
print("✅ IMPORT เสร็จสมบูรณ์!".center(70))
print("="*70)
print("\n🎉 พร้อมใช้งาน! เปิด main.ipynb ได้เลย")

# Show as DataFrame
df_summary = pd.DataFrame(summary)
display(df_summary)

## 🔌 Close Connection (Optional)

In [ ]:
# Uncomment to close
# cursor.close()
# connection.close()
# print("👋 Closed")